# Confidence Calibration & OOD Rejection — Run on Colab (experiment E5)

Final Year Project — Muhammad Asad Bukhari. **Runtime: CPU is enough**
(Runtime -> Change runtime type -> Hardware accelerator -> None).
Run the cells top to bottom: **Runtime -> Run all**. About 30–45 minutes.

**What this notebook does:** measures how well the disease model's
confidence scores mean what they say, and how well the model can tell
a real leaf from an out-of-distribution image:

1. Fits a **temperature** T on the validation logits (temperature
   scaling) and reports Expected Calibration Error (ECE) before/after.
2. Compares two **OOD signals** — max-softmax probability and the
   energy score — on validation leaves vs non-leaf photos, reports
   ROC AUC for both, and picks a rejection threshold from the ROC
   curve (95% TPR, same convention as experiment E4).
3. Compares the result with the old, unmeasured **80% rule**.

The model itself is never retrained or modified. The fitted
temperature and threshold are shipped to the app as constants.

**Outputs:** `results/calibration.json`, `results/reliability.png`,
`results/ood_roc.png`, `results/val_logits.npy`, `results/ood_logits.npy`.


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Devices:', tf.config.list_physical_devices())
print('GPUs:', tf.config.list_physical_devices('GPU'), '(CPU is fine here)')


In [ ]:
!git clone https://github.com/Asadshah-coder/leaf-disease-prediction.git
%cd leaf-disease-prediction
!pip install -q -r requirements.txt
!pip install -q gdown scikit-learn scipy
!ls


In [ ]:
# Disease model (86 MB) from Google Drive, file ID 1ufCFp8jhSFG6MMkcgF3XJsxm98y_7AjK — skip if already present.
import os
if not os.path.isfile('trained_plant_disease_model.keras'):
    get_ipython().system('gdown --id 1ufCFp8jhSFG6MMkcgF3XJsxm98y_7AjK -O trained_plant_disease_model.keras')
get_ipython().system('ls -lh trained_plant_disease_model.keras')


In [ ]:
# Dataset: PlantVillage leaves (1.7 GB) from Google Drive — skip if already extracted.
import os
if not os.path.isdir('data/valid'):
    get_ipython().system('gdown --id 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx -O "Plant Disease FYP.zip"')
    get_ipython().system('unzip -q "Plant Disease FYP.zip"')
    get_ipython().system('mkdir -p data && mv "Plant Disease FYP/train" data/train && mv "Plant Disease FYP/valid" data/valid')
get_ipython().system('echo "valid images:"; find data/valid -type f | wc -l')


In [ ]:
# OOD images: reuse the E4 gate's non-leaf validation set if present,
# otherwise sample 760 non-leaf images from Caltech 101 (plant categories excluded).
import os, random, shutil
random.seed(42)
os.makedirs('data/ood', exist_ok=True)
PLANT_CATEGORIES = {'bonsai', 'joshua-tree', 'lotus', 'strawberry', 'sunflower', 'water-lilly'}
if os.path.isdir('data/gate/val/nonleaf') and len(os.listdir('data/gate/val/nonleaf')) >= 700:
    for f in os.listdir('data/gate/val/nonleaf'):
        shutil.copy(os.path.join('data/gate/val/nonleaf', f), os.path.join('data/ood', f))
    print('OOD source: E4 gate validation set')
else:
    import subprocess
    if not os.path.isdir('101_ObjectCategories'):
        subprocess.run(['curl', '-L', '-o', 'caltech-101.zip',
                        'https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip'])
        get_ipython().system('unzip -q caltech-101.zip')
        get_ipython().system('find . -maxdepth 2 -name "101_ObjectCategories.tar.gz" -exec tar -xzf {} \;')
    pool = []
    for cls in sorted(os.listdir('101_ObjectCategories')):
        if cls in PLANT_CATEGORIES or cls.startswith('.'):
            continue
        d = os.path.join('101_ObjectCategories', cls)
        if not os.path.isdir(d):
            continue
        pool.extend([os.path.join(d, f) for f in os.listdir(d)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    random.shuffle(pool)
    for i, f in enumerate(pool[:760]):
        shutil.copy(f, os.path.join('data/ood', f'{i:05d}.jpg'))
    print('OOD source: Caltech 101, pool:', len(pool))
print('OOD images:', len(os.listdir('data/ood')))


In [ ]:
# Run the calibration + OOD analysis — about 10-20 minutes on a Colab CPU runtime.
# (Forward passes over 17,572 validation images + 760 OOD images; no training.)
!python src/calibrate.py --model trained_plant_disease_model.keras --data-dir data/valid --ood-dir data/ood --batch-size 64


In [ ]:
import json
print(open('results/calibration.json').read())
!ls -lh results/calibration.json results/reliability.png results/ood_roc.png


## Done — next steps

1. **Download `results/calibration.json`** from the Files panel, plus
   `results/reliability.png` and `results/ood_roc.png` (figures for
   the paper).
2. **Copy the numbers** into the experiment log
   `docs/experiments/E5.md`: temperature T, ECE before/after, OOD
   AUCs, the chosen signal and threshold, and the old-80%-rule
   comparison — only quote numbers from this run.
3. **Wire the fitted values into the app** (`main.py`): the temperature
   for calibrated top-3 confidences, and the measured threshold
   replacing the old 80% rule. Push, redeploy, and re-test live.
4. **Push** the notebook + experiment log to GitHub; **never** commit
   the `.keras` file, the dataset, or the `.npy` logit files (all are
   in `.gitignore` — check before pushing).
